In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from data_pipeline import cleaning_data

In [2]:
df=pd.read_csv(r"C:\Users\thien\code\AIMY\house-prices-advanced-regression-techniques\train.csv")
target_name='SalePrice'
drop_threshold=0.6

In [3]:
df=df.drop(columns='Id')

In [4]:
X,y=cleaning_data(df,target_name=target_name,drop_threshold=drop_threshold)

In [5]:
X.isna().any(axis=0).sum()

np.int64(0)

In [6]:
X.columns

Index(['MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'LotShape',
       'LandContour', 'LotConfig', 'Neighborhood', 'Condition1', 'BldgType',
       'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
       'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinSF2',
       'BsmtUnfSF', 'TotalBsmtSF', 'Heating', 'HeatingQC', 'CentralAir',
       'Electrical', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath',
       'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr',
       'KitchenQual', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageFinish', 'GarageCars', 'GarageQual', 'GarageCond', 'PavedDrive',
       'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch',
       'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold', 'SaleType',
       'SaleC

In [7]:
from pandasql import sqldf

In [8]:
pysqldf = lambda q: sqldf(q, globals())

In [ ]:
# ==============================================================================
# QUESTION 2 – Comparing Average Prices Between Remodeled and Non-Remodeled Homes
# ==============================================================================
# Objective : Measure the impact of remodeling on log(SalePrice).
#             Group homes by age range to examine how the remodeling premium
#             varies across different house ages.
# Result    : Remodeled homes aged 20–60 years have noticeably higher
#             avg_log_price, while for newer homes (< 20 years old),
#             remodeling provides little to no additional premium.
q2 = pysqldf("""

SELECT
        age_bucket,
        remod_flag,
        COUNT(*)                            AS so_nha,
        ROUND(AVG(SalePrice), 4)            AS avg_log_price,
        ROUND(MIN(SalePrice), 4)            AS min_log_price,
        ROUND(MAX(SalePrice), 4)            AS max_log_price,
        ROUND(AVG(SalePrice) - LAG(AVG(SalePrice))
              OVER (PARTITION BY age_bucket
                    ORDER BY remod_flag), 4) AS remod_premium_log
    FROM (
        SELECT
            SalePrice,
            CASE
                WHEN (YrSold - YearBuilt) < 20 THEN '0-19 yrs'
                WHEN (YrSold - YearBuilt) < 40 THEN '20-39 yrs'
                WHEN (YrSold - YearBuilt) < 60 THEN '40-59 yrs'
                ELSE '60+ yrs'
            END AS age_bucket,
            CASE
                WHEN YearRemodAdd > YearBuilt THEN 'Remodeled'
                ELSE 'Original'
            END AS remod_flag
        FROM df
    ) t
    GROUP BY age_bucket, remod_flag
    ORDER BY age_bucket, remod_flag
""")
print("\n[Q2] Tác động remodel lên log(SalePrice) theo nhóm tuổi nhà:")
print(q2.to_string(index=False))




[Q2] Tác động remodel lên log(SalePrice) theo nhóm tuổi nhà:
age_bucket remod_flag  so_nha  avg_log_price  min_log_price  max_log_price  remod_premium_log
  0-19 yrs   Original     346    227483.9422        84500.0       745000.0                NaN
  0-19 yrs  Remodeled     213    255280.9343        93500.0       755000.0         27796.9921
 20-39 yrs   Original     208    151715.0769        75000.0       320000.0                NaN
 20-39 yrs  Remodeled      55    190699.0909        86000.0       385000.0         38984.0140
 40-59 yrs   Original     210    139179.6952        55993.0       375000.0                NaN
 40-59 yrs  Remodeled     125    156695.6800        35311.0       335000.0         17515.9848
   60+ yrs  Remodeled     303    132675.8449        34900.0       475000.0                NaN


In [ ]:
# ==============================================================================
# QUESTION 3 – Ranking Neighborhoods by Average Price and Price Variability
# ==============================================================================
# Objective : Rank each neighborhood based on average log price (avg_log_price)
#             and measure price dispersion using standard deviation (std) to
#             identify neighborhoods with consistent prices (low std) versus
#             highly variable prices (high std).
# Result    : NoRidge and NridgHt have the highest average prices, while
#             OldTown and BrkSide exhibit the greatest price variability.
q3 = pysqldf("""
    SELECT
        Neighborhood,
        COUNT(*)                         AS so_nha,
        ROUND(AVG(SalePrice), 4)         AS avg_log_price,
        ROUND(MIN(SalePrice), 4)         AS min_log_price,
        ROUND(MAX(SalePrice), 4)         AS max_log_price,
        ROUND(MAX(SalePrice)
            - MIN(SalePrice), 4)         AS range_log_price
    FROM df
    GROUP BY Neighborhood
    ORDER BY avg_log_price DESC
""")
print("\n[Q3] Ranking Neighborhood theo log(SalePrice):")
print(q3.to_string(index=False))


[Q3] Ranking Neighborhood theo log(SalePrice):
Neighborhood  so_nha  avg_log_price  min_log_price  max_log_price  range_log_price
     NoRidge      41    335295.3171       190000.0       755000.0         565000.0
     NridgHt      77    316270.6234       154000.0       611657.0         457657.0
     StoneBr      25    310499.0000       170000.0       556581.0         386581.0
      Timber      38    242247.4474       137500.0       378500.0         241000.0
     Veenker      11    238772.7273       162500.0       385000.0         222500.0
     Somerst      86    225379.8372       144152.0       423000.0         278848.0
     ClearCr      28    212565.4286       130000.0       328000.0         198000.0
     Crawfor      51    210624.7255        90350.0       392500.0         302150.0
     CollgCr     150    197965.7733       110000.0       424870.0         314870.0
     Blmngtn      17    194870.8824       159895.0       264561.0         104666.0
     Gilbert      79    192854.5063    

In [ ]:
# ==============================================================================
# QUESTION 4 – Analyzing House Prices by Garage Capacity (GarageCars) and
#              Overall Quality (OverallQual)
# ==============================================================================
# Objective : Examine the interaction effect between GarageCars and
#             OverallQual on house prices.
#             Only groups with at least 10 houses are included to ensure
#             reliable results.
# Result    : The combination of GarageCars = 3 and OverallQual ≥ 8 achieves
#             the highest avg_log_price, confirming that an interaction term
#             (quality × garage capacity) is a valuable feature to engineer.
q4 = pysqldf("""
    SELECT
        GarageCars,
        OverallQual,
        COUNT(*)                         AS so_nha,
        ROUND(AVG(SalePrice), 4)         AS avg_log_price,
        ROUND(MIN(SalePrice), 4)         AS min_log_price,
        ROUND(MAX(SalePrice), 4)         AS max_log_price
    FROM df
    WHERE GarageCars IS NOT NULL
      AND OverallQual IS NOT NULL
    GROUP BY GarageCars, OverallQual
    HAVING COUNT(*) >= 10
    ORDER BY avg_log_price DESC
    LIMIT 15
""")
print("\n[Q4] Top combos GarageCars × OverallQual theo avg log(SalePrice):")
print(q4.to_string(index=False))



[Q4] Top combos GarageCars × OverallQual theo avg log(SalePrice):
 GarageCars  OverallQual  so_nha  avg_log_price  min_log_price  max_log_price
          3           10      16    463099.4375       184750.0       755000.0
          3            9      35    384701.9143       285000.0       611657.0
          3            8      80    302233.4000       147000.0       538000.0
          2            8      86    252254.6279       164000.0       430000.0
          3            7      38    244152.8421       140000.0       383970.0
          2            7     258    204785.6667        82500.0       375000.0
          1            7      20    176787.5000       116900.0       266500.0
          2            6     264    172819.7197        76000.0       277000.0
          2            5     163    144033.8037        55993.0       228950.0
          1            6      95    134987.6737        83000.0       259500.0
          1            5     195    127179.2769        60000.0       225000